In [20]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")


In [21]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from rich import print
import uuid

In [22]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from typing import Annotated
from operator import add
from typing import Literal

In [23]:
model = ChatOpenAI(
    model ='gpt-4.1-mini',
    api_key = api_key,
    temperature=0
)

In [27]:
class State(BaseModel):
    messages: Annotated[list[BaseMessage], add] = Field(default_factory=list)
    sender_email_address:str|None = Field(description="This field contains the email address of the sender", default=None)
    recipient_email_address:str| None = Field( description ="This field contains the email address of the reciever", default=None)
    subject:str| None = Field(description="The field contains the subject of the email", default=None)
    email_body: str | None = Field( description="This field contains the email that needs which is supposed to be sent", default=None)

In [ ]:
@tool
def save_email_draft(
    sender_email_address: str,
    recipient_email_address: str,
    subject: str,
    email_body: str,
) -> str:
    """Save an email draft when the user explicitly asks to save it."""
    return "Email draft saved."

In [29]:
@tool
def send_email(
    sender_email_address: str,
    recipient_email_address: str,
    subject: str,
    email_body: str,
) -> str:
    """Send an email after it has been approved by a human."""
    return "Email approved and sent."

In [30]:
tools = [save_email_draft, send_email]
model_with_tools = model.bind_tools(tools)
tool_by_name = {tool.name: tool for tool in tools}

system_prompt = """
You are a helpful email assistant.
Keep the conversation context and collect the sender, recipient, subject, and body.
Call save_email_draft when the user explicitly asks to save the draft. Also call it
when the user says the email conversation is finished, complete, or they are done.
Ask for any missing fields before calling save_email_draft.
Only call send_email after a draft has been saved. The graph will show the saved
details to a human for confirmation immediately before sending.
"""

In [31]:
EMAIL_FIELDS = (
    "sender_email_address",
    "recipient_email_address",
    "subject",
    "email_body",
)


def call_model(state: State):
    response = model_with_tools.invoke(
        [SystemMessage(content=system_prompt), *state.messages]
    )
    return {"messages": [response]}


def run_tools(state: State):
    updates = {"messages": []}
    latest_message = state.messages[-1]

    for tool_call in latest_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        if tool_name == "save_email_draft":
            approval = interrupt(
                {
                    "action": "save_email_draft",
                    "email": tool_args,
                    "message": "Approve saving these email details?",
                }
            )
            if not approval.get("approved", False):
                updates["messages"].append(
                    ToolMessage(
                        content="Draft was not saved.",
                        tool_call_id=tool_call["id"],
                    )
                )
                continue

            tool_result = tool_by_name[tool_name].invoke(tool_args)
            updates["messages"].append(
                ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
            )
            for field in EMAIL_FIELDS:
                updates[field] = tool_args[field]
            continue

        saved_email = {field: getattr(state, field) for field in EMAIL_FIELDS}
        if any(value is None for value in saved_email.values()):
            updates["messages"].append(
                ToolMessage(
                    content="No saved draft is available. Save the draft before sending.",
                    tool_call_id=tool_call["id"],
                )
            )
            continue

        approval = interrupt(
            {
                "action": "confirm_send_email",
                "email": saved_email,
                "message": "Approve sending this saved email?",
            }
        )
        if not approval.get("approved", False):
            updates["messages"].append(
                ToolMessage(
                    content="Email was not sent.",
                    tool_call_id=tool_call["id"],
                )
            )
            continue

        tool_result = tool_by_name[tool_name].invoke(saved_email)
        updates["messages"].append(
            ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
        )

    return updates


def route_after_model(state: State):
    return "tools" if state.messages[-1].tool_calls else END

In [ ]:
builder = StateGraph(State)
builder.add_node("model", call_model)
builder.add_node("tools", run_tools)
builder.add_edge(START, "model")
builder.add_conditional_edges("model", route_after_model, {"tools": "tools", END: END})
builder.add_edge("tools", "model")

email_graph = builder.compile(checkpointer=InMemorySaver())

In [ ]:
def chat_with_model(user_query: str, thread_id: str = "email-chat-1"):
    config = {"configurable": {"thread_id": thread_id}}
    return email_graph.invoke(
        {"messages": [HumanMessage(content=user_query)]},
        config=config,
    )


def respond_to_hitl(approved: bool, thread_id: str = "email-chat-1"):
    config = {"configurable": {"thread_id": thread_id}}
    return email_graph.invoke(
        Command(resume={"approved": approved}),
        config=config,
    )

In [ ]:
# Keep the same thread_id for every turn in one conversation.
# result = chat_with_model("Draft an email to Alice about tomorrow's meeting.")
# result = chat_with_model("Use a formal tone and add a clear subject.")
#
# Explicit save, or "I am done", causes a save HITL interrupt:
# result = chat_with_model("Save this draft now.")
# saved_result = respond_to_hitl(approved=True)
#
# Sending shows the saved values in a second HITL interrupt:
# result = chat_with_model("Send the saved email now.")
# sent_result = respond_to_hitl(approved=True)